# Clase 6: Archivos y Manejo de Errores

**Diplomado en Data Science Aplicada con Python para la Toma de Decisiones**  
Arca Continental Ecuador · UDLA · 2026

---

**Instructor:** Carlos Enrique Mosquera Trujillo  
**Contacto:** cmosquerat@unal.edu.co

---

### Contenido de esta sesión

| # | Tema | Duración |
|---|------|----------|
| 1 | Pipeline de Distribución (pendiente clase 5) | 12 min |
| 2 | Los tres tipos de error | 10 min |
| 3 | `try/except` — código que no explota | 15 min |
| 4 | Importar módulos | 10 min |
| 5 | Archivos CSV | 5 min |
| 6 | pandas — DataFrames | 25 min |
| 7 | Proyecto integrador | 15 min |

---
## 1. Pipeline de Distribución

Resolvamos el ejercicio pendiente de la clase pasada.

**El reto:** crear tres funciones para analizar rendimiento de vendedores.

1. `calcular_bono(vendedor, tasa=0.10)` — Si ventas ≥ meta → bono = ventas × tasa. Si no → 0. Retorna dict con clave `"bono"`.
2. `clasificar_vendedor(vendedor)` — Retorna `"Estrella"` si ventas ≥ 200, `"Cumple"` si ≥ meta, `"Bajo"` si no.
3. `reporte_equipo(lista, tasa=0.10)` — Recorre la lista, aplica ambas funciones, imprime una línea por vendedor y el total de bonos.

In [ ]:
vendedores = [
    {"nombre": "Ana",  "ventas": 250, "meta": 150},
    {"nombre": "Luis", "ventas": 90,  "meta": 150},
    {"nombre": "Sol",  "ventas": 180, "meta": 150},
    {"nombre": "Paco", "ventas": 310, "meta": 200}
]

In [ ]:
# Espacio para resolver

def calcular_bono(vendedor, tasa=0.10):
    pass  # TODO

def clasificar_vendedor(vendedor):
    pass  # TODO

def reporte_equipo(lista, tasa=0.10):
    pass  # TODO

reporte_equipo(vendedores)

In [ ]:
# Solución

def calcular_bono(vendedor, tasa=0.10):
    if vendedor["ventas"] >= vendedor["meta"]:
        vendedor["bono"] = vendedor["ventas"] * tasa
    else:
        vendedor["bono"] = 0
    return vendedor

def clasificar_vendedor(vendedor):
    if vendedor["ventas"] >= 200:
        return "Estrella"
    elif vendedor["ventas"] >= vendedor["meta"]:
        return "Cumple"
    else:
        return "Bajo"

def reporte_equipo(lista, tasa=0.10):
    total_bonos = 0
    for v in lista:
        calcular_bono(v, tasa)
        estado = clasificar_vendedor(v)
        print(f"{v['nombre']}: {estado}, bono={v['bono']}")
        total_bonos += v["bono"]
    print(f"Total bonos: {total_bonos}")

reporte_equipo(vendedores)

---
## 2. Los Tres Tipos de Error

¿Por qué mi código no funciona?

| Tipo | Descripción | Cuándo ocurre |
|------|-------------|---------------|
| **Sintaxis** | Escribiste algo que Python no entiende | El programa **no arranca** |
| **Runtime (Ejecución)** | La sintaxis está bien, pero al correr pides algo imposible | **Durante** la ejecución |
| **Lógico** | El programa corre sin errores, pero el resultado está **mal** | El más **difícil** de detectar |

In [ ]:
# Error de SINTAXIS — falta los dos puntos
def ingreso(cajas, precio)
    return cajas * precio

In [ ]:
# Error de RUNTIME — división por cero
def promedio(lista):
    return sum(lista) / len(lista)

promedio([])  # ZeroDivisionError

In [ ]:
# Error LÓGICO — suma en vez de multiplicar
def ingreso(cajas, precio):
    return cajas + precio  # Bug: debería ser *

print("Resultado:", ingreso(30, 12.50))  # Da 42.5 en vez de 375.0
print("Esperado:  375.0")

### Quiz: ¿Qué tipo de error es?

```python
datos = {"nombre": "Ana", "ventas": 250}
print(datos["meta"])  # ???
```

> **Respuesta:** Runtime — `KeyError`, la clave `"meta"` no existe en el diccionario.

---
## 3. `try/except` — Código que no explota

### ¿Cómo funciona?

Python intenta ejecutar el bloque `try`. Si algo falla, salta al `except` correspondiente. Si todo sale bien, ejecuta `else`. `finally` se ejecuta **siempre**, haya error o no.

In [ ]:
def leer_precio(texto):
    try:
        precio = float(texto)
    except ValueError:
        print(f"'{texto}' no es número")
        return None
    else:
        print(f"Precio válido: {precio}")
        return precio
    finally:
        print("--- Proceso terminado ---")

leer_precio("12.50")
print()
leer_precio("abc")

### La red de seguridad

- `try`: "intenta ejecutar este código"
- `except ZeroDivisionError`: "si falla con ESE error, haz esto"
- El programa **no se detiene**. Sigue ejecutando lo que viene después.

In [ ]:
# Sin protección — el programa explota
def promedio_sin(lista):
    return sum(lista) / len(lista)

# Con try/except — el programa sigue
def promedio_con(lista):
    try:
        return sum(lista) / len(lista)
    except ZeroDivisionError:
        return 0

print("Con protección:", promedio_con([]))
print("El programa sigue normalmente")

### `except` específico vs genérico

**Regla de oro:** cada `except` debe nombrar el error que espera. Si no sabes cuál va a pasar, deja que explote — así aprendes qué es.

In [ ]:
# ❌ Genérico — oculta bugs
try:
    resultado = 100 / 0
except:
    print("Algo falló")
    resultado = 0

print("---")

# ✅ Específico — sabes qué pasó
try:
    resultado = 100 / 0
except ZeroDivisionError:
    print("Error: división por cero")
    resultado = 0
except KeyError as e:
    print(f"Clave no encontrada: {e}")

### `try/except` en funciones reales

Combinamos validación (clase 5) + `try/except`: usar `try/except` para errores de datos y `if` para reglas de negocio.

In [ ]:
def procesar_venta(vendedor):
    try:
        nombre = vendedor["nombre"]
        ventas = vendedor["ventas"]
        meta = vendedor["meta"]
    except KeyError as e:
        return {"error": f"Falta el campo {e}"}

    if ventas < 0:
        return {"error": "Ventas negativas"}

    bono = ventas * 0.10 if ventas >= meta else 0
    return {"nombre": nombre, "bono": bono, "sobre_meta": ventas >= meta}

print(procesar_venta({"nombre": "Ana", "ventas": 250, "meta": 150}))
print(procesar_venta({"nombre": "Luis", "ventas": 90}))
print(procesar_venta({"nombre": "Sol", "ventas": -5, "meta": 150}))

### Práctica: Blindar con `try/except` (7 min)

Modifiquen esta función para que **nunca explote**.

**Salida esperada:**
```
{'nombre': 'Ana', 'comision': 25.0}
{'error': "Falta campo 'meta'"}
{'error': 'Tasa debe estar entre 0 y 1'}
```

In [ ]:
def calcular_comision(vendedor, tasa=0.10):
    # TODO: envolver en try/except KeyError
    nombre = vendedor["nombre"]
    ventas = vendedor["ventas"]
    meta   = vendedor["meta"]

    # TODO: validar que tasa esté entre 0 y 1

    if ventas >= meta:
        return {"nombre": nombre, "comision": ventas * tasa}
    return {"nombre": nombre, "comision": 0}

# Pruebas
print(calcular_comision({"nombre": "Ana", "ventas": 250, "meta": 150}))
print(calcular_comision({"nombre": "Luis", "ventas": 90}))
print(calcular_comision({"nombre": "Sol", "ventas": 180, "meta": 150}, tasa=5))

In [ ]:
# Solución

def calcular_comision(vendedor, tasa=0.10):
    try:
        nombre = vendedor["nombre"]
        ventas = vendedor["ventas"]
        meta   = vendedor["meta"]
    except KeyError as e:
        return {"error": f"Falta campo {e}"}

    if not (0 <= tasa <= 1):
        return {"error": "Tasa debe estar entre 0 y 1"}

    if ventas >= meta:
        return {"nombre": nombre, "comision": ventas * tasa}
    return {"nombre": nombre, "comision": 0}

print(calcular_comision({"nombre": "Ana", "ventas": 250, "meta": 150}))
print(calcular_comision({"nombre": "Luis", "ventas": 90}))
print(calcular_comision({"nombre": "Sol", "ventas": 180, "meta": 150}, tasa=5))

---
## 4. Importar Módulos

El superpoder de Python: **no reinventar la rueda.**

Un módulo es un archivo `.py` con código reutilizable — funciones, clases y variables que alguien ya escribió y probó. Tú solo los importas y usas.

### Librería estándar (incluida con Python)

| Módulo | Uso |
|--------|-----|
| `math` | `sqrt`, `pi`, `factorial` |
| `random` | Números aleatorios |
| `datetime` | Fechas y horas |
| `os` | Archivos y sistema |
| `json` | Leer/escribir JSON |

### Librerías externas (se instalan con `pip install`)

| Librería | Uso |
|----------|-----|
| `pandas` | Tablas de datos (DataFrames) |
| `numpy` | Cálculo numérico |
| `matplotlib` | Gráficas y visualización |
| `scikit-learn` | Machine Learning |
| `requests` | Descargar datos de la web |

### Tres formas de importar

- `import X` → usas `X.funcion()`. Mantiene el código claro.
- `from X import Y` → usas `Y()` directo. Más corto pero menos explícito.
- `import X as alias` → `import pandas as pd` — la convención en Data Science.

In [ ]:
# 1. Importar el módulo completo
import math
print("Raíz de 144:", math.sqrt(144))
print("Pi:", math.pi)

# 2. Importar solo lo que necesitas
from math import factorial
print("5! =", factorial(5))

# 3. Importar con alias
import datetime as dt
hoy = dt.date.today()
print("Hoy:", hoy)

# Bono: combinar
from random import randint
print("Dado:", randint(1, 6))

---
## 5. Archivos CSV

### ¿Qué es un archivo CSV?

**CSV = Comma Separated Values** — un archivo de texto plano donde cada línea es una fila y las columnas se separan por comas.

Es el formato más universal para intercambiar datos tabulares:
- Excel, Google Sheets, bases de datos — todos exportan e importan CSV.
- La primera fila son los encabezados (nombres de columna).

```
nombre,ventas,meta,zona
Ana,18500,15000,Norte
Luis,9200,15000,Sur
Sol,22000,15000,Norte
Paco,15000,15000,Sur
Mia,31000,20000,Norte
Carlos,12800,15000,Centro
Diana,27500,20000,Sur
Felipe,8500,12000,Centro
Rosa,19000,15000,Norte
Tomas,16500,15000,Centro
```

---
## 6. pandas — La herramienta #1 para datos en Python

**pandas = Excel dentro de Python.**

Una librería que te da **DataFrames**: tablas con filas y columnas, pero con todo el poder de Python para filtrar, agrupar y transformar datos.

- Una línea para cargar un CSV.
- Los tipos de dato se detectan automáticamente.
- Filtros, agrupaciones y estadísticas integradas.
- Usado en el 95% de proyectos de Data Science.

**Convención universal:** `import pandas as pd` — siempre.

In [ ]:
# Crear el archivo CSV de ejemplo
csv_data = """nombre,ventas,meta,zona
Ana,18500,15000,Norte
Luis,9200,15000,Sur
Sol,22000,15000,Norte
Paco,15000,15000,Sur
Mia,31000,20000,Norte
Carlos,12800,15000,Centro
Diana,27500,20000,Sur
Felipe,8500,12000,Centro
Rosa,19000,15000,Norte
Tomas,16500,15000,Centro"""

with open("datos_ventas.csv", "w") as f:
    f.write(csv_data)

print("Archivo datos_ventas.csv creado.")

In [ ]:
import pandas as pd

df = pd.read_csv("datos_ventas.csv")
print("Tipo:", type(df))
print("Forma:", df.shape)
print()
df

### Explorar el DataFrame

| Método | Uso |
|--------|-----|
| `.head(n)` | Primeras n filas |
| `.tail(n)` | Últimas n filas |
| `.shape` | (filas, columnas) |
| `.columns` | Nombres de columnas |
| `.dtypes` | Tipos de dato |
| `.describe()` | Estadísticas resumen |
| `.info()` | Resumen del DataFrame |

In [ ]:
import pandas as pd
df = pd.read_csv("datos_ventas.csv")

print("=== head() ===")
print(df.head(3))

print("\nColumnas:", list(df.columns))

print("\n=== describe() ===")
print(df.describe())

### Seleccionar y filtrar datos

- `df["col"]` → una columna. `df[["a","b"]]` → varias columnas.
- `df[condición]` → filtra filas. Combinar condiciones con `&` y `|`.

In [ ]:
import pandas as pd
df = pd.read_csv("datos_ventas.csv")

print("=== Nombres ===")
print(df["nombre"])

# Crear columna nueva
df["cumple_meta"] = df["ventas"] >= df["meta"]
print("\n=== Con nueva columna ===")
print(df)

# Filtrar filas
sobre_meta = df[df["ventas"] >= df["meta"]]
print(f"\n=== {len(sobre_meta)} vendedores sobre meta ===")
print(sobre_meta[["nombre", "ventas", "meta"]])

### Agrupar y resumir: `groupby()`

Equivalente a una **tabla dinámica de Excel**, pero en una línea de código.

| Función | Resultado |
|---------|-----------|
| `.sum()` | Suma total |
| `.mean()` | Promedio |
| `.count()` | Cantidad de filas |
| `.min()` / `.max()` | Mínimo / Máximo |
| `.median()` | Mediana |
| `.std()` | Desviación estándar |

In [ ]:
import pandas as pd
df = pd.read_csv("datos_ventas.csv")

por_zona = df.groupby("zona")["ventas"].agg(["sum", "mean", "count"])
print("=== Ventas por zona ===")
print(por_zona)

print(f"\nTotal ventas: ${df['ventas'].sum():,}")
print(f"Promedio: ${df['ventas'].mean():,.0f}")

### Exportar: `df.to_csv()`

- `to_csv(ruta, index=False)` — siempre `index=False` para no exportar el índice.
- Pipeline estándar: `read_csv()` → transformar → `to_csv()`.

In [ ]:
import pandas as pd
df = pd.read_csv("datos_ventas.csv")

df["comision"] = df.apply(
    lambda v: v["ventas"] * 0.05 if v["ventas"] >= v["meta"] else 0,
    axis=1
)

df["estado"] = df.apply(
    lambda v: "Estrella" if v["ventas"] >= 2 * v["meta"]
              else "Cumple" if v["ventas"] >= v["meta"]
              else "Bajo",
    axis=1
)

df.to_csv("reporte.csv", index=False)

print(pd.read_csv("reporte.csv").to_string())

### Práctica: Analizar con pandas (10 min)

**Pistas:**
- `df["col"] >= df["col2"]` para comparar.
- `df[~condicion]` para negar.
- `df.groupby("zona")["ventas"].mean()` para promedios.
- `df.loc[df["ventas"].idxmax()]` para el máximo.

In [ ]:
import pandas as pd
df = pd.read_csv("datos_ventas.csv")

# TODO: Crear columna "cumple" (True/False)

# TODO: Filtrar los que NO cumplen meta

# TODO: Calcular promedio de ventas por zona

# TODO: Mostrar el vendedor con más ventas

print(df)

In [ ]:
# Solución

import pandas as pd
df = pd.read_csv("datos_ventas.csv")

# 1. Crear columna "cumple"
df["cumple"] = df["ventas"] >= df["meta"]
print("=== DataFrame con 'cumple' ===")
print(df)

# 2. Filtrar los que NO cumplen meta
no_cumplen = df[~df["cumple"]]
print(f"\n=== {len(no_cumplen)} no cumplen meta ===")
print(no_cumplen[["nombre", "ventas", "meta"]])

# 3. Promedio de ventas por zona
print("\n=== Promedio por zona ===")
print(df.groupby("zona")["ventas"].mean())

# 4. Vendedor con más ventas
top = df.loc[df["ventas"].idxmax()]
print(f"\nTop vendedor: {top['nombre']} con ${top['ventas']:,}")

---
## 7. Proyecto Integrador

### Pipeline completo con pandas: CSV → análisis → reporte

**Escenario:** El gerente regional les pide un script que cargue el CSV de ventas, calcule comisiones, clasifique vendedores, genere un resumen por zona y exporte el resultado.

**Pasos:**
1. `pd.read_csv()` para cargar datos.
2. Crear columna `"comision"` — <meta→0%, ≥meta→5%, ≥2×meta→10%.
3. Crear columna `"estado"` — "Estrella" / "Cumple" / "Bajo".
4. `groupby("zona")` para resumen por zona.
5. `to_csv()` para exportar el reporte.

| Condición | Comisión |
|-----------|----------|
| ventas < meta | 0% |
| ventas ≥ meta | 5% |
| ventas ≥ 2 × meta | 10% |

In [ ]:
import pandas as pd

# 1. Cargar datos
df = pd.read_csv("datos_ventas.csv")

# 2. TODO: Crear columna "comision"
#    <meta→0%, >=meta→5%, >=2*meta→10%

# 3. TODO: Crear columna "estado"
#    "Estrella" / "Cumple" / "Bajo"

# 4. TODO: Mostrar reporte por consola
#    print(df[["nombre","zona","estado","comision"]])

# 5. TODO: Resumen por zona con groupby

# 6. TODO: Exportar a CSV
#    df.to_csv("reporte_final.csv", index=False)

print(df)

In [ ]:
# Solución

import pandas as pd

df = pd.read_csv("datos_ventas.csv")

# Comisión
def calc_comision(row):
    if row["ventas"] >= 2 * row["meta"]:
        return row["ventas"] * 0.10
    elif row["ventas"] >= row["meta"]:
        return row["ventas"] * 0.05
    return 0

df["comision"] = df.apply(calc_comision, axis=1)

# Estado
def clasificar(row):
    if row["ventas"] >= 2 * row["meta"]:
        return "Estrella"
    elif row["ventas"] >= row["meta"]:
        return "Cumple"
    return "Bajo"

df["estado"] = df.apply(clasificar, axis=1)

# Reporte
print("=== Reporte de Ventas ===")
print(df[["nombre", "zona", "estado", "comision"]].to_string(index=False))

# Resumen por zona
print("\n=== Resumen por Zona ===")
resumen = df.groupby("zona")[["ventas", "comision"]].sum()
print(resumen)
print(f"\nTotal comisiones: ${df['comision'].sum():,.0f}")

# Exportar
df.to_csv("reporte_final.csv", index=False)
print("\nArchivo reporte_final.csv exportado.")

---
## Resumen: Lo que aprendimos hoy

- Pipeline completo con funciones (clase 05).
- **Tres tipos de error:** sintaxis, runtime, lógica.
- Leer mensajes de error **de abajo hacia arriba**.
- `try/except` para código que no explota.
- `except` específico vs genérico.
- **Módulos:** estándar vs externos, `pip install`.
- Tres formas de importar: `import`, `from`, alias.
- Qué es un CSV y por qué importa.
- **pandas:** `read_csv`, filtrar, `groupby`, `to_csv`.
- Pipeline completo: CSV → pandas → reporte.

### Próxima clase

- pandas avanzado — merge, pivot tables, datos faltantes.
- Visualización con matplotlib — gráficas de datos.
- Conectar datos reales de la empresa.